In [2]:
pip install matplotlib

  Using cached matplotlib-3.10.9-cp310-cp310-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp310-cp310-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp310-cp310-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.9-cp310-cp310-win_amd64.whl (8.2 MB)
Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl (221 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp310-cp310-win_amd64.whl (1.6 MB)
Using cached kiwisolver-1.5.0-cp310-cp310-win_amd64.whl (73 kB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)

   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- --

In [3]:
#!/usr/bin/env python3
"""
Batch measurement script: loads masks and raw images, measures per-cell properties,
applies size-based filtering, and generates QC histograms and overview images.
 
Changes vs original 02:
  - DAPI (channel 0) is always read and used to estimate tissue area via Otsu
    thresholding on the max-intensity projection (MIP).  The tissue area (µm²)
    is written to both the per-image props CSV and the summary CSV.
  - Far-red (channel 3) intensity is always measured for every valid green cell
    (no longer skipped when the channel is absent — an explicit error is raised
    instead, keeping the pipeline honest).
  - Summary CSV gains columns: tissue_area_um2, n_valid_per_um2.
"""
 
import os
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import tifffile
from typing import List
import skimage
import skimage.transform
import skimage.segmentation
import skimage.filters
import skimage.measure
import skimage.exposure
import skimage.util
import skimage.morphology
 
# --- Paths ---
DEFAULT_INPUT = r"N:\Sarah\Croatia project\DAPI, 488 NeuN, 594 AT8_hippocampus"
INPUT_ROOT = Path(os.environ.get("SEG_INPUT_ROOT", DEFAULT_INPUT))
MASK_ROOT      = INPUT_ROOT / "masks_cellpose_sam_3d_Algernon"
HISTOGRAM_ROOT = INPUT_ROOT / "histograms"
OVERVIEWS_ROOT = INPUT_ROOT / "overviews"
 
MASK_SUFFIX = "_mask_3d"
 
# Channels inside the .ims file
CH_DAPI   = 0   # blue  — used for tissue-area estimation
CH_GREEN  = 1   # NeuN  — segmentation channel; valid cells defined here
CH_RED    = 2   # AT8
CH_FARRED = 3   # WFA (or whatever far-red marker is in use)
 
# Size filter thresholds (µm³) — same as original
MIN_AREA = 300
MAX_AREA = 4000
 
 
# ---------------------------------------------------------------------------
# I/O helpers
# ---------------------------------------------------------------------------
 
def read_ims_channels(path, channel_indices):
    """Read selected channels from an Imaris .ims (HDF5) file.
 
    Returns
    -------
    data : np.ndarray  shape (len(channel_indices), Z, Y, X)
    xy_spacing : float  physical pixel size in XY (µm)
    z_spacing  : float  physical pixel size in Z  (µm)
    """
    with h5py.File(path, "r") as f:
        ds_root = f["DataSet"]["ResolutionLevel 0"]["TimePoint 0"]
        channels = []
        for ci in channel_indices:
            raw = ds_root[f"Channel {ci}"]["Data"][:]
            channels.append(raw)
        data = np.stack(channels, axis=0)  # (C, Z, Y, X)
 
        info = f["DataSetInfo"]["Image"].attrs
 
        def _attr(key):
            v = info.get(key, None)
            if v is None:
                return None
            if isinstance(v, np.ndarray):
                if v.dtype.kind == 'S':
                    v = b''.join(v.flatten().tolist()).decode()
                elif v.dtype.kind in ('U', 'O'):
                    v = ''.join(str(x.decode() if isinstance(x, bytes) else x) for x in v.flatten())
                else:
                    return float(v.flat[0])
            if isinstance(v, (bytes, bytearray, np.bytes_)):
                v = v.decode()
            return float(str(v).strip())
 
        x_min, x_max = _attr("ExtMin0"), _attr("ExtMax0")
        z_min, z_max = _attr("ExtMin2"), _attr("ExtMax2")
        nx = data.shape[3]
        nz = data.shape[1]
        xy_spacing = (x_max - x_min) / nx if (x_min is not None and x_max is not None) else 1.0
        z_spacing  = (z_max - z_min) / nz if (z_min is not None and z_max is not None) else 1.0
 
    return data, xy_spacing, z_spacing
 
 
def find_mask_files(root: Path, token: str) -> List[Path]:
    files: List[Path] = []
    for path in root.rglob("*.tif"):
        if token not in path.name.lower():
            continue
        files.append(path)
    return sorted(files)
 
 
def _to_uint8(arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32)
    lo, hi = 50, 1000
    arr = (arr - lo) / (hi - lo) * 255
    return np.clip(arr, 0, 255).astype(np.uint8)
 
 
def save_histogram(values, parameter_name, name, out_dir, bins=100):
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    plt.figure(figsize=(7, 4.5))
    plt.hist(values, bins=bins, edgecolor="black", alpha=0.85)
    plt.title(f"Histogram: {name}")
    plt.xlabel(parameter_name)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(out_path / f"{name}_histogram.png", dpi=200)
    plt.close()
 
 
# ---------------------------------------------------------------------------
# Tissue-area estimation
# ---------------------------------------------------------------------------
 
def estimate_tissue_area_um2(dapi_volume_zyx: np.ndarray, xy_spacing: float) -> float:
    """
    Project DAPI along Z (max-intensity projection), threshold with Otsu,
    and return the tissue footprint area in µm².
 
    The MIP collapses depth, so each pixel represents xy_spacing² µm².
    """
    mip = dapi_volume_zyx.max(axis=0).astype(np.float32)   # (Y, X)
    # Normalise to uint16 range for a stable Otsu calculation
    mip_norm = skimage.exposure.rescale_intensity(mip, out_range=(0, 65535)).astype(np.uint16)
    thresh = skimage.filters.threshold_otsu(mip_norm)
    tissue_mask = mip_norm > thresh
    n_tissue_pixels = int(tissue_mask.sum())
    return n_tissue_pixels * (xy_spacing ** 2)

In [5]:
pip install napari[all]

  Using cached app_model-0.5.1-py3-none-any.whl.metadata (3.2 kB)
  Using cached cachey-0.2.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached magicgui-0.10.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached napari_console-0.1.4-py3-none-any.whl.metadata (5.7 kB)
  Using cached napari_plugin_engine-0.2.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached napari_svg-0.2.1-py3-none-any.whl.metadata (3.6 kB)
  Using cached npe2-0.8.2-py3-none-any.whl.metadata (6.8 kB)
  Using cached psygnal-0.15.1-cp310-cp310-win_amd64.whl.metadata (6.1 kB)
  Using cached pyopengl-3.1.10-py3-none-any.whl.metadata (3.3 kB)
  Using cached superqt-0.8.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached vispy-0.16.2-cp310-cp310-win_amd64.whl.metadata (9.2 kB)
  Using cached in_n_out-0.2.1-py3-none-any.whl.metadata (4.1 kB)
  Using cached freetype_py-2.5.1-py3-none-win_amd64.whl.metadata (6.5 kB)
  Using cached hsluv-5.0.4-py2.py3-none-any.wh

In [6]:
import napari

In [9]:
mask_path = r"L:\Algernon\530\Sarah\segmentation_output_pseudo3d_260611\motor cortex\64U-J9910_CHST11KO_MC_AT8_delta4_first_2026-04-23_12.12.05_FusionStitcher_mask_3d.tif"
image_path = r"N:\01_scientific_data\Sarah_Kralova_brain_slice_data\cellpose + far-red channel\motor cortex\64U-J9910_CHST11KO_MC_AT8_delta4_first_2026-04-23_12.12.05_FusionStitcher.ims"


In [ ]:


# --- 3. Load all four channels (DAPI, green, red, far-red) ---
image_data, xy_spacing, z_spacing = read_ims_channels(
    image_path,
    channel_indices=[CH_DAPI, CH_GREEN, CH_RED, CH_FARRED],
)
# image_data shape: (4, Z, Y, X) in the order requested above
dapi_channel   = image_data[0]
green_channel  = image_data[1]
red_channel    = image_data[2]
farred_channel = image_data[3]

print(f"  xy_spacing={xy_spacing:.4f} µm, z_spacing={z_spacing:.4f} µm")
print(f"  image shape (CZYX): {image_data.shape}")


In [10]:

# --- 1. Load segmentation mask ---
mask_data = tifffile.imread(str(mask_path)).squeeze()   # (Z, Y, X)
print(f"  mask shape: {mask_data.shape}")

# --- 4. Crop all channels to mask shape if needed ---
if mask_data.shape != green_channel.shape:
    mz, my, mx = mask_data.shape
    iz, iy, ix = green_channel.shape
    if iz < mz or iy < my or ix < mx:
        raise ValueError(
            f"Image {green_channel.shape} is smaller than mask {mask_data.shape} — cannot crop."
        )
    print(f"  NOTE: cropping image from {green_channel.shape} to {mask_data.shape}")
    dapi_channel   = dapi_channel[:mz, :my, :mx]
    green_channel  = green_channel[:mz, :my, :mx]
    red_channel    = red_channel[:mz, :my, :mx]
    farred_channel = farred_channel[:mz, :my, :mx]


viewer = napari.Viewer()
viewer.add_image(green_channel, name="Green")
viewer.add_labels(mask_data, name="Mask")


  mask shape: (12, 4546, 4658)


<Labels layer 'Mask' at 0x29dbcc10430>

In [8]:
viewer.add_image(mask_data, name="Mask", colormap="red", opacity=0.5)

<Image layer 'Mask' at 0x29dd8cf7a00>

In [ ]:
# --- 3. Load all four channels (DAPI, green, red, far-red) ---
image_data, xy_spacing, z_spacing = read_ims_channels(
    image_path,
    channel_indices=[CH_DAPI, CH_GREEN, CH_RED, CH_FARRED],
)
# image_data shape: (4, Z, Y, X) in the order requested above
dapi_channel   = image_data[0]
green_channel  = image_data[1]
red_channel    = image_data[2]
farred_channel = image_data[3]

print(f"  xy_spacing={xy_spacing:.4f} µm, z_spacing={z_spacing:.4f} µm")
print(f"  image shape (CZYX): {image_data.shape}")

In [ ]:
mask_path = r"L:\Algernon\530\Sarah\segmentation_output_pseudo3d_260611\motor cortex\64U-J9910_CHST11KO_MC_AT8_delta4_first_2026-04-23_12.12.05_FusionStitcher_input_channel.tif"
image_path = r"N:\01_scientific_data\Sarah_Kralova_brain_slice_data\cellpose + far-red channel\motor cortex\64U-J9910_CHST11KO_MC_AT8_delta4_first_2026-04-23_12.12.05_FusionStitcher.ims"

# --- 1. Load segmentation mask ---
mask_data = tifffile.imread(str(mask_path)).squeeze()   # (Z, Y, X)
print(f"  mask shape: {mask_data.shape}")

# --- 4. Crop all channels to mask shape if needed ---
if mask_data.shape != green_channel.shape:
    mz, my, mx = mask_data.shape
    iz, iy, ix = green_channel.shape
    if iz < mz or iy < my or ix < mx:
        raise ValueError(
            f"Image {green_channel.shape} is smaller than mask {mask_data.shape} — cannot crop."
        )
    print(f"  NOTE: cropping image from {green_channel.shape} to {mask_data.shape}")
    dapi_channel   = dapi_channel[:mz, :my, :mx]
    green_channel  = green_channel[:mz, :my, :mx]
    red_channel    = red_channel[:mz, :my, :mx]
    farred_channel = farred_channel[:mz, :my, :mx]

# --- 5. Tissue area from DAPI ---
tissue_area_um2 = estimate_tissue_area_um2(dapi_channel, xy_spacing)
print(f"  Tissue area (Otsu on DAPI MIP): {tissue_area_um2:.1f} µm²")

# --- 6. Measure region properties ---
spacing = (z_spacing, xy_spacing, xy_spacing)
props_green  = skimage.measure.regionprops(mask_data, green_channel,  spacing=spacing)
props_red    = skimage.measure.regionprops(mask_data, red_channel,    spacing=spacing)
props_farred = skimage.measure.regionprops(mask_data, farred_channel, spacing=spacing)

areas = np.array([p.area for p in props_green])
valid = (areas >= MIN_AREA) & (areas <= MAX_AREA)

# --- 7. Build per-cell dataframe ---
def _mean_map(props_list):
    return {p.label: getattr(p, 'intensity_mean', np.nan) for p in props_list}

def _std_map(props_list):
    return {p.label: getattr(p, 'intensity_std', np.nan) for p in props_list}

red_mean    = _mean_map(props_red)
red_std     = _std_map(props_red)
farred_mean = _mean_map(props_farred)
farred_std  = _std_map(props_farred)

labels = [p.label for p in props_green]

# Red and far-red intensities are recorded only for valid green cells;
# invalid cells get NaN so they are cleanly excluded from downstream analysis.
props_df = pd.DataFrame({
    "label":                  labels,
    "area":                   [p.area          for p in props_green],
    "num_pixels":             [p.num_pixels     for p in props_green],
    "intensity_mean_green":   [p.intensity_mean for p in props_green],
    "intensity_mean_red":     [red_mean.get(l, np.nan)    if v else np.nan for l, v in zip(labels, valid)],
    "intensity_std_red":      [red_std.get(l, np.nan)     if v else np.nan for l, v in zip(labels, valid)],
    "intensity_mean_farred":  [farred_mean.get(l, np.nan) if v else np.nan for l, v in zip(labels, valid)],
    "intensity_std_farred":   [farred_std.get(l, np.nan)  if v else np.nan for l, v in zip(labels, valid)],
    "valid":                  valid,
    # Tissue area repeated per row so the props CSV is self-contained
    "tissue_area_um2":        tissue_area_um2,
})


# --- 9. Size-filtered mask → MIP overlay ---
invalid_labels     = props_df.loc[~valid, "label"].to_numpy()
filtered_mask_data = mask_data.copy()
filtered_mask_data[np.isin(filtered_mask_data, invalid_labels)] = 0

green_mip = green_channel.max(axis=0)
red_mip   = red_channel.max(axis=0)
mask_mip  = filtered_mask_data.max(axis=0).astype(np.int32)

scale = 1 / 8
green_mip_s = skimage.transform.rescale(green_mip, scale, anti_aliasing=True,  preserve_range=True)
red_mip_s   = skimage.transform.rescale(red_mip,   scale, anti_aliasing=True,  preserve_range=True)
mask_mip_s  = skimage.transform.rescale(mask_mip,  scale, anti_aliasing=False, order=0, preserve_range=True).astype(np.int32)

boundaries = skimage.segmentation.find_boundaries(mask_mip_s, mode="outer")
rgb        = np.zeros((*green_mip_s.shape, 3), dtype=np.uint8)
rgb[..., 0] = _to_uint8(red_mip_s)
rgb[..., 1] = _to_uint8(green_mip_s)
rgb[boundaries] = [255, 255, 255]

OVERVIEWS_ROOT.mkdir(parents=True, exist_ok=True)
plt.imsave(OVERVIEWS_ROOT / f"{image_name}_mip_overlay.png", rgb)

# --- 10. Per-image summary row ---
n_masks       = len(props_green)
n_valid_masks = int(valid.sum())
density       = n_valid_masks / tissue_area_um2 if tissue_area_um2 > 0 else float("nan")

threshold_otsu     = skimage.filters.threshold_otsu(areas)
threshold_triangle = skimage.filters.threshold_triangle(areas)

records.append({
    "image_name":         image_name,
    "n_masks":            n_masks,
    "n_valid_masks":      n_valid_masks,
    "tissue_area_um2":    tissue_area_um2,
    "n_valid_per_um2":    density,
    "threshold_otsu":     threshold_otsu,
    "threshold_triangle": threshold_triangle,
    "xy_spacing":         xy_spacing,
    "z_spacing":          z_spacing,
})

# --- 11. Summary CSV ---
df = pd.DataFrame(records)
out_csv = HISTOGRAM_ROOT / "summary.csv"
df.to_csv(out_csv, index=False)
print(f"\nSummary saved to {out_csv}")
print("Done.")
